### Hipótesis 1 -- Entrenar solo con el mes más cercano (agosto)

Si los datos cambian mucho de un mes a otro (drift), el mes más cercano alcanzaría. Mismo procedimiento que el baseline Val-B2 (z803); solo cambian las ventanas: optimización con train 201906 y validación 201908, reentrenamiento con 201908 y test en 201910.

In [1]:
# Configuración de recursos según la máquina
entorno = 'casa'  # cambiar a 'oficina' según corresponda

if entorno == 'casa':
    # i7-12700K, 12 cores, 32GB RAM
    duckdb_threads = 8
    duckdb_memory_limit = '16GB'
    rf_n_jobs = 8
    lgb_threads = 8
elif entorno == 'oficina':
    # 16 cores, 63.8GB RAM
    duckdb_threads = 12
    duckdb_memory_limit = '45GB'
    rf_n_jobs = 10
    lgb_threads = 10          # 8 por precaución: la oficina se apagó con CPU al 100% el 27/09
else:
    raise ValueError("entorno debe ser 'casa' u 'oficina'")

print(f"Entorno: {entorno} | DuckDB: {duckdb_threads} threads, {duckdb_memory_limit} | RF: n_jobs={rf_n_jobs} | LGB: {lgb_threads} threads")

Entorno: casa | DuckDB: 8 threads, 16GB | RF: n_jobs=8 | LGB: 8 threads


#### Librerías y Path Variables

In [2]:
# Importando Librerías

import os, sys
os.environ['PATH'] = os.path.join(sys.prefix, 'Library', 'bin') + os.pathsep + os.environ['PATH']
import duckdb

import pandas as pd
import numpy as np
import pickle                                                                   # CAMBIO

import plotly.graph_objects as go
import plotly.express as px

from sklearn.metrics import roc_auc_score                                       # CAMBIO

import lightgbm as lgb                                                          # CAMBIO

import optuna                                                                   # CAMBIO
from optuna.visualization import plot_optimization_history, plot_param_importances, plot_slice, plot_contour   # CAMBIO

In [3]:
# Paths Variables

from pathlib import Path
nb_file = globals().get('__vsc_ipynb_file__')                 # ruta de la notebook (la define VS Code)
cwd = Path(nb_file).parent if nb_file else Path.cwd()         # carpeta de la notebook; si no es VS Code, usa el cwd
RAIZ = (cwd / '../../../../05_ExpMeses').resolve()

sys.path.insert(0, str(cwd))                                  # semillas.py está al lado de la notebook
from semillas import SEMILLAS as semillas

base_path = RAIZ.as_posix() + '/'                             # as_posix: barras normales también en Windows
print(RAIZ, RAIZ.exists(), (cwd / 'semillas.py').exists())

dataset_path = base_path + 'data/'
db_path = base_path + 'databases/'
modelos_path = base_path + 'modelos/'
out_path = base_path + 'out/'
dataset_file = 'competencia_02_meses.parquet'
ruta_meses = dataset_path + dataset_file
print(ruta_meses, Path(ruta_meses).exists())

os.makedirs(db_path, exist_ok=True)                            # CAMBIO: carpeta de las bases de Optuna
os.makedirs(modelos_path, exist_ok=True)                       # CAMBIO: carpeta de modelos

E:\Training\05_Exactas-UBA\06_DM_EconomiaFinanzas\05_ExpMeses True True
E:/Training/05_Exactas-UBA/06_DM_EconomiaFinanzas/05_ExpMeses/data/competencia_02_meses.parquet True


#### Carga de los Meses
Se leen de `competencia_02_meses.parquet` los meses 201901 a 201910 y se dejan en una tabla en memoria de DuckDB (`d2019`). 201911 y 201912 son del dashboard y no se usan. Se cargan todos los meses para calcular los lags.

In [4]:
%%time
# Lee del parquet de meses; solo hasta 201910 (201911 y 201912 son del dashboard)
con = duckdb.connect()
con.execute(f"SET threads = {duckdb_threads}")
con.execute(f"SET memory_limit = '{duckdb_memory_limit}'")
con.execute("SET enable_progress_bar = false")

MES_DESDE, MES_HASTA = 201901, 201910

con.execute(f"""
    CREATE OR REPLACE TABLE d2019 AS
    SELECT * FROM read_parquet('{ruta_meses}')
    WHERE foto_mes BETWEEN {MES_DESDE} AND {MES_HASTA}
""")
print(con.execute("SELECT count(*) FROM d2019").fetchall()[0][0], 'filas')

1286487 filas
CPU times: total: 5.64 s
Wall time: 1.08 s


#### Filas, Clientes, Variables y Clase Ternaria por Mes

In [5]:
mes_resumen = con.sql("""
    SELECT foto_mes,
           count(*)                                      AS filas,
           count(DISTINCT numero_de_cliente)             AS clientes,
           count(*) - count(DISTINCT numero_de_cliente)  AS duplicados      -- tiene que dar 0
    FROM d2019
    GROUP BY foto_mes
    ORDER BY foto_mes
""").df()
mes_resumen

,foto_mes,filas,clientes,duplicados
0,201901,123916,123916,0
1,201902,125020,125020,0
2,201903,125622,125622,0
3,201904,126116,126116,0
4,201905,126763,126763,0
5,201906,128370,128370,0
6,201907,129853,129853,0
7,201908,131775,131775,0
8,201909,133419,133419,0
9,201910,135633,135633,0


In [6]:
import math

cols = [r[0] for r in con.execute("SELECT column_name FROM (DESCRIBE d2019)").fetchall()]
print(len(cols), 'columnas')
print('clase_ternaria' in cols)

n_col = 5                                          # cantidad de columnas a mostrar
n_fil = math.ceil(len(cols) / n_col)
relleno = cols + [''] * (n_fil * n_col - len(cols))     # completa con vacíos para que cierre el rectángulo
tabla = pd.DataFrame(np.array(relleno).reshape(n_col, n_fil).T)
tabla.style.hide(axis='index').hide(axis='columns')

155 columnas
True


numero_de_cliente,mtarjeta_master_consumo,mpagodeservicios,mcheques_emitidos_rechazados,Master_mpagado
foto_mes,cprestamos_personales,cpagomiscuentas,tcallcenter,Master_mpagospesos
active_quarter,mprestamos_personales,mpagomiscuentas,ccallcenter_transacciones,Master_mpagosdolares
cliente_vip,cprestamos_prendarios,ccajeros_propios_descuentos,thomebanking,Master_fechaalta
internet,mprestamos_prendarios,mcajeros_propios_descuentos,chomebanking_transacciones,Master_mconsumototal
cliente_edad,cprestamos_hipotecarios,ctarjeta_visa_descuentos,ccajas_transacciones,Master_cconsumos
cliente_antiguedad,mprestamos_hipotecarios,mtarjeta_visa_descuentos,ccajas_consultas,Master_cadelantosefectivo
mrentabilidad,cplazo_fijo,ctarjeta_master_descuentos,ccajas_depositos,Master_mpagominimo
mrentabilidad_annual,mplazo_fijo_dolares,mtarjeta_master_descuentos,ccajas_extracciones,Visa_delinquency
mcomisiones,mplazo_fijo_pesos,ccomisiones_mantenimiento,ccajas_otras,Visa_status
mactivos_margen,cinversion1,mcomisiones_mantenimiento,catm_trx,Visa_mfinanciacion_limite


In [7]:
# Clase por mes. Esperado: 201911 solo con BAJA+1, 201912 sin clase
clase_mes = (con.sql("""
    SELECT foto_mes, coalesce(clase_ternaria, '(sin clase)') AS clase, count(*) AS filas
    FROM d2019
    GROUP BY 1, 2
    ORDER BY 1, 2
""").df().pivot(index='foto_mes', columns='clase', values='filas').fillna(0).astype(int))

total = clase_mes.sum(axis=1)
clase_mes['pct_BAJA+2'] = (clase_mes.get('BAJA+2', 0) / total * 100).round(3)
clase_mes

clase,BAJA+1,BAJA+2,CONTINUA,pct_BAJA+2
foto_mes,,,,
201901,647,697,122572,0.562
201902,700,696,123624,0.557
201903,700,774,124148,0.616
201904,776,575,124765,0.456
201905,576,658,125529,0.519
201906,665,589,127116,0.459
201907,595,656,128602,0.505
201908,659,543,130573,0.412
201909,548,615,132256,0.461


#### Tipos de Variables

In [8]:
esquema = con.sql("DESCRIBE d2019").df()[['column_name', 'column_type']]
ids = ['numero_de_cliente', 'foto_mes']

tipos_num = ('TINYINT', 'SMALLINT', 'INTEGER', 'BIGINT', 'HUGEINT', 'UTINYINT', 'USMALLINT', 'UINTEGER', 'UBIGINT',
             'FLOAT', 'DOUBLE', 'DECIMAL')
es_num = esquema['column_type'].str.startswith(tipos_num)

cols_num = [c for c in esquema.loc[es_num, 'column_name'] if c not in ids]            # a las que se les mide nulos y ceros
cols_otras = [c for c in esquema.loc[~es_num, 'column_name'] if c not in ids]         # texto, fechas, etc.

print(len(esquema), 'columnas |', len(cols_num), 'numéricas |', len(cols_otras), 'no numéricas')
print('No numéricas:', cols_otras)           # clase_ternaria y, si aparecen, columnas 0/1 que llegaron como texto
esquema['column_type'].value_counts()

155 columnas | 152 numéricas | 1 no numéricas
No numéricas: ['clase_ternaria']


column_type
BIGINT     80
DOUBLE     74
VARCHAR     1
Name: count, dtype: int64

#### Feature Engineering -- Lags y Deltas 1 y 2

In [9]:
%%time
# Feature engineering: lag 1 y 2 y delta 1 y 2 de las variables numéricas, calculados por mes real
ruta_fe = dataset_path + 'competencia_02_fe.parquet'        # parquet de salida (no es el de entrada, para no pisarlo)
forzar_fe = False                                           # CAMBIO: True para recalcular aunque el parquet ya exista

if Path(ruta_fe).exists() and not forzar_fe:                # CAMBIO: si ya está calculado, no se repite
    print('ya existe, no se recalcula:', ruta_fe, round(Path(ruta_fe).stat().st_size / 1e6), 'MB')
else:
    # Índice de mes continuo (año*12 + mes): permite restar 1 o 2 y cruzar de diciembre a enero sin saltos
    mes_idx = "(foto_mes // 100 * 12 + foto_mes % 100)"
    cols_sql = ', '.join(f'"{c}"' for c in cols_num)            # las 152 numéricas, entre comillas, para el SELECT

    # Una columna nueva por variable y tipo: lag_1, lag_2 (valor de hace 1 y 2 meses) y delta_1, delta_2 (valor actual menos el lag)
    sel = []
    for c in cols_num:
        sel.append(f'p1."{c}" AS "lag_1_{c}"')
        sel.append(f'p2."{c}" AS "lag_2_{c}"')
        sel.append(f'd."{c}" - p1."{c}" AS "delta_1_{c}"')
        sel.append(f'd."{c}" - p2."{c}" AS "delta_2_{c}"')

    query = f"""
        -- base: la tabla de 2019 con el índice de mes; prev: el mismo cliente y mes, solo con las variables a "mirar para atrás"
        WITH base AS (SELECT *, {mes_idx} AS mes_idx FROM d2019),
             prev AS (SELECT numero_de_cliente, mes_idx, {cols_sql} FROM base)
        -- d.*: todas las columnas originales (sin el índice auxiliar) más los lags y deltas
        SELECT d.* EXCLUDE (mes_idx), {', '.join(sel)}
        FROM base d
        -- mismo cliente, un mes antes (p1) y dos meses antes (p2); LEFT JOIN: si no existe ese mes, el lag queda nulo
        LEFT JOIN prev p1 ON p1.numero_de_cliente = d.numero_de_cliente AND p1.mes_idx = d.mes_idx - 1
        LEFT JOIN prev p2 ON p2.numero_de_cliente = d.numero_de_cliente AND p2.mes_idx = d.mes_idx - 2
    """
    # Se escribe directo a parquet (comprimido con ZSTD), con orden fijo de filas
    con.execute(f"""
        COPY ({query} ORDER BY d.foto_mes, d.numero_de_cliente)
        TO '{ruta_fe}' (FORMAT PARQUET, COMPRESSION ZSTD)
    """)
    print('guardado:', ruta_fe, round(Path(ruta_fe).stat().st_size / 1e6), 'MB')

ya existe, no se recalcula: E:/Training/05_Exactas-UBA/06_DM_EconomiaFinanzas/05_ExpMeses/data/competencia_02_fe.parquet 785 MB
CPU times: total: 0 ns
Wall time: 288 μs


In [10]:
# 1) filas, clientes y columnas (esperadas: las de d2019 + 4 por cada variable numérica)
n_fil, n_cli = con.execute(f"SELECT count(*), count(DISTINCT numero_de_cliente) FROM read_parquet('{ruta_fe}')").fetchall()[0]
n_col = len(con.execute(f"DESCRIBE SELECT * FROM read_parquet('{ruta_fe}')").fetchall())
print(n_fil, 'filas |', n_cli, 'clientes |', n_col, 'columnas | esperadas:', len(esquema) + 4 * len(cols_num))

# 2) % de nulos de los lags de una variable, por mes
con.execute(f"""
    SELECT foto_mes,
           round(100 * avg((lag_1_mcuentas_saldo IS NULL)::int), 2) AS pct_null_lag1,
           round(100 * avg((lag_2_mcuentas_saldo IS NULL)::int), 2) AS pct_null_lag2
    FROM read_parquet('{ruta_fe}') GROUP BY 1 ORDER BY 1
""").df()

1286487 filas | 141458 clientes | 763 columnas | esperadas: 763


,foto_mes,pct_null_lag1,pct_null_lag2
0,201901,100.00,100.00
1,201902,1.40,100.00
2,201903,1.04,2.43
3,201904,0.95,1.98
4,201905,1.12,2.06
5,201906,1.70,2.81
6,201907,1.65,3.33
7,201908,1.91,3.53
8,201909,1.73,3.61
9,201910,2.04,3.72


##### Cargado de Datos para LightGBM

In [ ]:
%%time
# Carga de los meses de la hipótesis 1 desde el parquet de FE                     
meses_opt_train  = [201906]                             # optimización: train solo con junio (el mes más cercano a la validación)
meses_opt_val    =  201908                              # optimización: validación
meses_test_train = [201908]                             # reentrenamiento para el test solo con agosto (el mes más cercano al test)
meses_test       =  201910                              # test

MESES = sorted(set(meses_opt_train + [meses_opt_val] + meses_test_train + [meses_test]))
lista_meses = ', '.join(str(m) for m in MESES)

data = con.execute(f"""
    SELECT * FROM read_parquet('{ruta_fe}')
    WHERE foto_mes IN ({lista_meses})
    ORDER BY foto_mes, numero_de_cliente
""").fetchdf()                                           # orden fijo de filas

print(data.shape)

# Filas por mes
print('\nFilas por mes:')
print(data['foto_mes'].value_counts().sort_index().to_string())

# Resumen de los bloques de meses, con sus filas
def filas(meses):
    meses = meses if isinstance(meses, list) else [meses]
    return int(data['foto_mes'].isin(meses).sum())

bloques = {
    'Optimización | train':      meses_opt_train,
    'Optimización | validación': meses_opt_val,
    'Test         | train':      meses_test_train,
    'Test         | test':       meses_test,
}
print('\nBloques de meses:')
for nombre, meses in bloques.items():
    print(f'{nombre:27s}: {meses}  ->  {filas(meses):,} filas')

(395778, 763)

Filas por mes:
foto_mes
201906    128370
201908    131775
201910    135633

Bloques de meses:
Optimización | train       : [201906]  ->  128,370 filas
Optimización | validación  : 201908  ->  131,775 filas
Test         | train       : [201908]  ->  131,775 filas
Test         | test        : 201910  ->  135,633 filas
CPU times: total: 20.5 s
Wall time: 10.2 s


##### Clase Binaria + X/y de Optimización (train con BAJA+1 y BAJA+2, validación con BAJA+2)

In [12]:
# Clase binaria: BAJA+1 y BAJA+2 juntas (1) contra CONTINUA (0)
print('Filas sin clase:', int(data['clase_ternaria'].isna().sum()))      # tiene que dar 0 en estos meses
data['clase_binaria_b12'] = np.where(data['clase_ternaria'].isin(['BAJA+1', 'BAJA+2']), 1, 0)
data['clase_binaria_b2']  = np.where(data['clase_ternaria'].isin(['BAJA+2']), 1, 0)       # solo BAJA+2 (1) contra BAJA+1 y CONTINUA (0)

# No entran a X: la clase, las binarias, el id y el mes
cols_descartar = ['clase_ternaria', 'clase_binaria_b12', 'clase_binaria_b2', 'numero_de_cliente', 'foto_mes']

# Optimización: train (mes 6) y validación (mes 8)                       # CAMBIO: train solo con junio
train_data = data[data['foto_mes'].isin(meses_opt_train)]
val_data   = data[data['foto_mes'] == meses_opt_val]

X_train = train_data.drop(columns=cols_descartar)
y_train = train_data['clase_binaria_b12']                                 # train: target fusionado

X_val = val_data.drop(columns=cols_descartar)
y_val = val_data['clase_binaria_b2']                                      # validación solo con BAJA+2

# feature_pre_filter=False: necesario porque Optuna va a variar min_data_in_leaf entre trials
train_data_lgb = lgb.Dataset(X_train, label=y_train, params={'feature_pre_filter': False})
val_data_lgb   = lgb.Dataset(X_val, label=y_val, reference=train_data_lgb)

print('train:', X_train.shape, '| positivos', int(y_train.sum()), f'({100 * y_train.mean():.2f}%)')
print('valid:', X_val.shape,   '| positivos BAJA+2', int(y_val.sum()),   f'({100 * y_val.mean():.2f}%)')

Filas sin clase: 0
train: (128370, 760) | positivos 1254 (0.98%)
valid: (131775, 760) | positivos BAJA+2 543 (0.41%)


#### Optimización de Hiperparámetros (Optuna, Validación en 201908)

In [13]:
# Optimización bayesiana (Optuna): rangos del baseline.yaml, validación en el mes 201908
def objective(trial):

    params = {
        'objective': 'binary',
        'metric': 'auc',
        'boosting_type': 'gbdt',
        'boost_from_average': True,
        'feature_pre_filter': False,
      # bayesianos
        'num_leaves': trial.suggest_int('num_leaves', 8, 256),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 20, 5000, log=True),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.1, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.3, 1.0),
      # fijos
        'bagging_freq': 1,
        'seed': semillas[0],
        'num_threads': lgb_threads,
        'verbose': -1
    }

    model = lgb.train(
        params,
        train_data_lgb,
        num_boost_round=5000,                                           # techo; el corte lo pone el early stopping
        valid_sets=[val_data_lgb],                                      # validación en 201908 
        callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)]
    )

    trial.set_user_attr('best_iter', model.best_iteration)              # iteraciones del mejor AUC
    return model.best_score['valid_0']['auc']                           # AUC en validación: ordena los trials

storage_name = "sqlite:///" + db_path + "ExpMeses_Optimization.db"
study_name = "EM_H01_TrB12_ValB2_T201906V201908"                        # study propio de la hipótesis 1 (train 201906, validación 201908)

study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=semillas[0]),               # sampler con semilla, reproducible
    study_name=study_name,
    storage=storage_name,
    load_if_exists=True,
)

print(len(study.trials))     # debe dar 0 si el estudio es nuevo o dar 50 si el estudio ya existe en la base

[I 2026-11-19 22:38:24,961] A new study created in RDB with name: EM_H01_TrB12_ValB2_T201906V201908


0


In [14]:
objetivo_total = 50                                        # número de trials que queremos correr en total
faltan = objetivo_total - len(study.trials)
print(f"Tengo {len(study.trials)}, corro {faltan} más")
if faltan > 0:
    study.optimize(objective, n_trials=faltan)

Tengo 0, corro 50 más


[I 2026-11-19 22:39:25,449] Trial 0 finished with value: 0.8997837549616206 and parameters: {'num_leaves': 158, 'learning_rate': 0.01887605300918835, 'min_data_in_leaf': 76, 'feature_fraction': 0.22060412069722457, 'bagging_fraction': 0.5419355046340902}. Best is trial 0 with value: 0.8997837549616206.
[I 2026-11-19 22:39:38,788] Trial 1 finished with value: 0.901517389191784 and parameters: {'num_leaves': 93, 'learning_rate': 0.02168135018114355, 'min_data_in_leaf': 1909, 'feature_fraction': 0.27639724152867773, 'bagging_fraction': 0.49118686408473866}. Best is trial 1 with value: 0.901517389191784.
[I 2026-11-19 22:39:43,582] Trial 2 finished with value: 0.8966912252570118 and parameters: {'num_leaves': 66, 'learning_rate': 0.0655520385792004, 'min_data_in_leaf': 1041, 'feature_fraction': 0.23124508247802583, 'bagging_fraction': 0.5330431525187476}. Best is trial 1 with value: 0.901517389191784.
[I 2026-11-19 22:39:52,723] Trial 3 finished with value: 0.8997951710672912 and parameter

##### Resultados Optimización de Hiperparámetros

In [15]:
# Historia de la optimización: AUC de cada trial y mejor AUC acumulado
optuna.visualization.plot_optimization_history(study)

In [16]:
# Importancia de cada hiperparámetro: cuánto explica de las diferencias de AUC entre trials
plot_param_importances(study)

In [17]:
# Slice: AUC de cada trial según el valor de cada hiperparámetro (uno por gráfico)
plot_slice(study)

In [18]:
# Contour de un solo par: learning_rate contra min_data_in_leaf (el freno del baseline)
plot_contour(study, params=['learning_rate', 'min_data_in_leaf'])

##### Mejor Trial

In [19]:
mejor = study.best_trial
print('Mejor trial:', mejor.number, '| AUC validación:', round(mejor.value, 4), '| best_iter:', mejor.user_attrs['best_iter'])
mejor.params

Mejor trial: 41 | AUC validación: 0.9063 | best_iter: 367


{'num_leaves': 35,
 'learning_rate': 0.015320604677943266,
 'min_data_in_leaf': 1151,
 'feature_fraction': 0.3918995418311729,
 'bagging_fraction': 0.89958285077904}

#### Train 201908 y Test 201910 con 5 Semillas

##### Variables para Reentrenar y Testear

In [20]:
# Test: train con el mes 8 y test en el mes 10                                      # train solo con agosto
data_test_train = data[data['foto_mes'].isin(meses_test_train)]       # filas de 201908 (el mes de validación)
data_test       = data[data['foto_mes'] == meses_test]                # filas de 201910 (para testear el modelo final)

# X: variables sin la clase, el id y el mes (cols_descartar)
X_test_train = data_test_train.drop(columns=cols_descartar)
y_test_train = data_test_train['clase_binaria_b12']                   # train: target fusionado (BAJA+1 y BAJA+2 = 1, CONTINUA = 0)
X_test       = data_test.drop(columns=cols_descartar)
y_test = data_test['clase_ternaria']                                  # clase ternaria de octubre, solo para medir con el BAJA+2 real; no entra en X

# Dataset de LightGBM con el train del test; feature_pre_filter=False por consistencia con la optimización
data_test_train_lgb = lgb.Dataset(X_test_train, label=y_test_train, params={'feature_pre_filter': False})

# Chequeo: forma de cada matriz y cantidad de positivos
print('train:', X_test_train.shape, '| positivos (fusionado)', int(y_test_train.sum()), f'({100 * y_test_train.mean():.2f}%)')
print('test :', X_test.shape,       '| BAJA+2', int((y_test == 'BAJA+2').sum()), f'({100 * (y_test == "BAJA+2").mean():.2f}%)')

train: (131775, 760) | positivos (fusionado) 1202 (0.91%)
test : (135633, 760) | BAJA+2 632 (0.47%)


##### Reentrenamiento y Testeo con AUC y Ganancia

In [21]:
# Parámetros de la ganancia
ganancia_acierto = 1_072_500                              # ganancia por cliente BAJA+2 enviado
costo_estimulo   = 27_500                                 # costo por cliente enviado que no es BAJA+2

# Ganancia con el BAJA+2 real: se acierta solo si el cliente fue BAJA+2, aunque el modelo haya aprendido las dos bajas
def ganancia_real(y_pred, y_true_class, umbral=0.025):
    ganancia = np.where(y_true_class == 'BAJA+2', ganancia_acierto, -costo_estimulo)
    return ganancia[y_pred >= umbral].sum()

In [22]:
n_semillas_test = 5                                       # cantidad de semillas: sirve para medir cuánto varía la ganancia por azar
best_iter = mejor.user_attrs['best_iter']                 # iteraciones del mejor trial (sin validación no hay early stopping)
forzar_entrenamiento = False                              # True para reentrenar aunque el modelo ya esté guardado

rows = []                                                 # un resultado por semilla
pred_semillas = []                                        # la predicción de octubre de cada semilla, para el semillerío
for seed in semillas[:n_semillas_test]:                   # las primeras 5 semillas de semillas.py
    ruta_modelo = modelos_path + f'EM_H01_ValB2_Train08_Test10_s{seed}.txt'       # CAMBIO: modelos propios de la hipótesis 1

    if Path(ruta_modelo).exists() and not forzar_entrenamiento:       # si el modelo ya existe, se carga y no se entrena
        model = lgb.Booster(model_file=ruta_modelo)
        print(f'semilla {seed}: modelo cargado')
    else:
        params = {
            'objective': 'binary', 'metric': 'auc', 'boosting_type': 'gbdt', 'boost_from_average': True,
            'feature_pre_filter': False,
            **mejor.params,                               # hiperparámetros del mejor trial de la optimización
            'bagging_freq': 1,                            # fijo, como en la optimización
            'seed': seed,                                 # lo único que cambia entre las 5 corridas
            'num_threads': lgb_threads,
            'verbose': -1
        }
        # Entrena con 201908 (target fusionado) y las iteraciones del mejor trial; guarda el modelo de cada semilla     # CAMBIO: comentario
        model = lgb.train(params, data_test_train_lgb, num_boost_round=best_iter)
        model.save_model(ruta_modelo)
        print(f'semilla {seed}: modelo entrenado y guardado')

    y_pred = model.predict(X_test)                        # probabilidad de baja de cada cliente en octubre
    pred_semillas.append(y_pred)                          # se guarda para el semillerío
    rows.append({
        'seed': seed,
        'auc_test': roc_auc_score(data_test['clase_binaria_b2'], y_pred),   # AUC contra BAJA+2 solamente
        'ganancia': ganancia_real(y_pred, y_test),                          # ganancia con umbral 0.025, solo BAJA+2 gana (y_test es la clase ternaria)
    })

# Semillerío: probabilidad promedio de cada cliente en las 5 semillas
y_pred_prom = np.mean(pred_semillas, axis=0)
auc_prom = roc_auc_score(data_test['clase_binaria_b2'], y_pred_prom)
ganancia_prom = ganancia_real(y_pred_prom, y_test)        # ganancia del semillerío con umbral 0.025
rows.append({'seed': 'semillerio', 'auc_test': auc_prom, 'ganancia': ganancia_prom})

df_test = pd.DataFrame(rows)                              # una fila por semilla y una del semillerío

semilla 618541778: modelo entrenado y guardado
semilla 249611762: modelo entrenado y guardado
semilla 424089056: modelo entrenado y guardado
semilla 176108349: modelo entrenado y guardado
semilla 472407521: modelo entrenado y guardado


#### Resultados en formato común (5 semillas + semillerío)

Reúne los resultados del baseline en tablas con el mismo formato (`experimento`, `modelo`), para compararlos con las hipótesis. El semillerío es el promedio de las probabilidades de las 5 semillas y entra como un modelo más. Se guardan cuatro archivos en `out/`: `test`, `cortes`, `mejorN` y `curvas`.

In [23]:
experimento = 'H01_ValB2'                            # nombre del experimento: lo cambia cada hipótesis
limite = 20000                                            # cuántos clientes se guardan de la curva (igual que el gráfico)
cortes = np.arange(2000, 8001, 500)                       # cortes fijos de la tabla de ganancias

# Los 6 modelos: las 5 semillas y el semillerío, cada uno con su predicción de octubre
modelos_pred = [(str(seed), pred) for seed, pred in zip(semillas[:n_semillas_test], pred_semillas)]
modelos_pred.append(('semillerio', y_pred_prom))

rows_curvas, rows_cortes, rows_mejorN = [], [], []
for modelo, pred in modelos_pred:
    orden = np.argsort(pred)[::-1]                        # clientes de mayor a menor probabilidad
    ganancia_ordenada = np.where(y_test.values[orden] == 'BAJA+2', ganancia_acierto, -costo_estimulo)
    ganancia_acumulada = np.cumsum(ganancia_ordenada)     # ganancia si se estimulan los primeros 1, 2, 3... clientes

    # Curva: ganancia acumulada de los primeros 'limite' clientes
    rows_curvas.append(pd.DataFrame({
        'experimento': experimento, 'modelo': modelo,
        'clientes': np.arange(1, limite + 1),
        'ganancia_acumulada': ganancia_acumulada[:limite],
    }))

    # Cortes fijos: ganancia al estimular los primeros N clientes
    rows_cortes.append(pd.DataFrame({
        'experimento': experimento, 'modelo': modelo,
        'corte': cortes,
        'pct_clientes': cortes / len(pred) * 100,         # CAMBIO: el corte como porcentaje de los clientes de octubre
        'ganancia': ganancia_acumulada[cortes - 1],
    }))

    # Mejor corte: el N que maximiza la ganancia acumulada (usa las etiquetas de test: es un techo, no un corte que se pueda elegir de antemano)
    mejor_n = int(np.argmax(ganancia_acumulada[:limite])) + 1
    rows_mejorN.append({
        'experimento': experimento, 'modelo': modelo,
        'mejor_corte': mejor_n,
        'pct_clientes': mejor_n / len(pred) * 100,
        'ganancia_maxima': ganancia_acumulada[mejor_n - 1],
    })

resultados_curvas = pd.concat(rows_curvas, ignore_index=True)
resultados_cortes = pd.concat(rows_cortes, ignore_index=True)
resultados_mejorN = pd.DataFrame(rows_mejorN)

# Test con umbral 0.025: es df_test, con 'experimento' y 'modelo' (la columna seed pasa a modelo)
resultados_test = df_test.rename(columns={'seed': 'modelo'}).astype({'modelo': str})
resultados_test.insert(0, 'experimento', experimento)

# Guarda los cuatro archivos en out/
resultados_test.to_csv(out_path + f'{experimento}_test.csv', index=False)
resultados_cortes.to_csv(out_path + f'{experimento}_cortes.csv', index=False)
resultados_mejorN.to_csv(out_path + f'{experimento}_mejorN.csv', index=False)
resultados_curvas.to_csv(out_path + f'{experimento}_curvas.csv', index=False)

##### Tablas

In [24]:
from IPython.display import display

semillas_ids = list(resultados_test.loc[resultados_test['modelo'] != 'semillerio', 'modelo'])   # ids de las 5 semillas

# ---- Tabla 1: media ± desvío de las 5 semillas y semillerío ----
r = resultados_test.set_index('modelo')                                         # datos originales (ganancia sin pasar a millones)
s = r.loc[semillas_ids]                                                         # solo las 5 semillas (el semillerío no es una semilla más)
resumen = pd.DataFrame({
    '5 semillas (media ± desvío)': [
        f"{s['auc_test'].mean():.4f} ± {s['auc_test'].std():.4f}",
        f"{s['ganancia'].mean() / 1e6:.2f} ± {s['ganancia'].std() / 1e6:.2f}",   # CAMBIO: en millones, con dos decimales
    ],
    'semillerío': [                                                             # un solo modelo: sin desvío
        f"{r.loc['semillerio', 'auc_test']:.4f}",
        f"{r.loc['semillerio', 'ganancia'] / 1e6:.2f}",                          # CAMBIO: en millones, con dos decimales
    ],
}, index=['AUC test (BAJA+2)', 'Ganancia umbral 0.025 (M)'])                     # CAMBIO: la etiqueta indica millones
display(resumen.style.set_caption('Test 201910 - media ± desvío de las 5 semillas y semillerío'))

# ---- Tabla 2: test con umbral 0.025, por semilla ----
t = resultados_test.set_index('modelo')[['auc_test', 'ganancia']]
t.loc['mediana 5 semillas'] = t.loc[semillas_ids].median()
t = t.loc[semillas_ids + ['mediana 5 semillas', 'semillerio']]                  # orden: semillas, mediana, semillerío
t['ganancia'] = t['ganancia'] / 1e6                                             # ganancia en millones
t.columns = ['AUC test (BAJA+2)', 'Ganancia umbral 0.025 (M)']
display(t.style.format({'AUC test (BAJA+2)': '{:.4f}', 'Ganancia umbral 0.025 (M)': '{:.2f}'})
        .set_caption('Test 201910 - 5 semillas, mediana y semillerío'))

# ---- Tabla 3: ganancia en cada corte fijo (millones) ----
c = resultados_cortes.pivot(index='corte', columns='modelo', values='ganancia') / 1e6
pct = resultados_cortes[resultados_cortes['modelo'] == 'semillerio'].set_index('corte')['pct_clientes']   # % de clientes de cada corte
tabla_cortes = pd.DataFrame({
    '% clientes': pct,
    'mediana': c[semillas_ids].median(axis=1),
    'media': c[semillas_ids].mean(axis=1),
    'desvío': c[semillas_ids].std(axis=1),
    'mín': c[semillas_ids].min(axis=1),
    'máx': c[semillas_ids].max(axis=1),
    'semillerío': c['semillerio'],
})
display(tabla_cortes.style.format('{:.2f}')
        .set_caption('Ganancia en Octubre por corte (M) - estadísticas de las 5 semillas y semillerío'))

# ---- Tabla 4: mejor corte y ganancia máxima ----
m = resultados_mejorN.set_index('modelo')[['mejor_corte', 'pct_clientes', 'ganancia_maxima']]
m.loc['mediana 5 semillas'] = m.loc[semillas_ids].median()
m = m.loc[semillas_ids + ['mediana 5 semillas', 'semillerio']]                  # orden: semillas, mediana, semillerío
m['ganancia_maxima'] = m['ganancia_maxima'] / 1e6                               # ganancia en millones
m.columns = ['Mejor corte (clientes)', '% clientes', 'Ganancia máxima (M)']
display(m.style.format({'Mejor corte (clientes)': '{:,.0f}', '% clientes': '{:.2f}', 'Ganancia máxima (M)': '{:.2f}'})
        .set_caption('Mejor corte por modelo (usa las etiquetas de test: es un techo)'))

,5 semillas (media ± desvío),semillerío
AUC test (BAJA+2),0.9186 ± 0.0008,0.9198
Ganancia umbral 0.025 (M),191.52 ± 6.95,193.66


,AUC test (BAJA+2),Ganancia umbral 0.025 (M)
modelo,,
618541778,0.9175,182.68
249611762,0.9182,186.67
424089056,0.9195,196.76
176108349,0.9187,199.51
472407521,0.9194,191.98
mediana 5 semillas,0.9187,191.98
semillerio,0.9198,193.66


,% clientes,mediana,media,desvío,mín,máx,semillerío
corte,,,,,,,
2000,1.47,247.50,246.62,5.12,239.80,251.90,246.40
2500,1.84,263.45,263.01,7.96,250.25,271.15,270.05
3000,2.21,271.70,273.68,3.34,270.60,278.30,276.10
3500,2.58,277.75,277.53,3.68,273.35,282.15,277.75
4000,2.95,276.10,278.30,3.73,275.00,283.80,279.40
4500,3.32,277.75,276.21,3.07,271.15,278.85,278.85
5000,3.69,275.00,275.00,2.80,271.70,278.30,277.20
5500,4.06,271.15,272.69,3.78,268.95,277.75,273.35
6000,4.42,268.40,267.30,3.81,261.80,270.60,268.40


,Mejor corte (clientes),% clientes,Ganancia máxima (M)
modelo,,,
618541778,"4,282",3.16,279.35
249611762,"4,810",3.55,279.12
424089056,"3,600",2.65,279.40
176108349,"4,050",2.99,286.82
472407521,"4,317",3.18,281.68
mediana 5 semillas,"4,282",3.16,279.40
semillerio,"4,275",3.15,282.84


##### Curvas de Ganancia Acumulada

In [25]:
import plotly.graph_objects as go

fig = go.Figure()
for modelo, d in resultados_curvas.groupby('modelo', sort=False):
    es_semillerio = modelo == 'semillerio'
    fig.add_trace(go.Scatter(
        x=d['clientes'], y=d['ganancia_acumulada'], mode='lines',
        name='semillerío' if es_semillerio else f'semilla {modelo}',
        line=dict(color='black', width=3, dash='dash') if es_semillerio else dict(width=1.5),   # el semillerío se distingue
    ))

fig.update_layout(
    title=f"Ganancia acumulada en Octubre - {resultados_curvas['experimento'].iloc[0]}",
    xaxis_title='Cantidad de clientes ordenados por probabilidad descendente',
    yaxis_title='Ganancia acumulada',
    template='plotly_white'
)
fig.show()

##### Boxplot

In [26]:
fig = go.Figure()
d = resultados_cortes[resultados_cortes['modelo'] != 'semillerio']              # el boxplot es solo de las 5 semillas
for corte, dc in d.groupby('corte'):
    fig.add_trace(go.Box(y=dc['ganancia'], name=str(corte), showlegend=False, marker_color='steelblue'))

s = resultados_cortes[resultados_cortes['modelo'] == 'semillerio']               # el semillerío, un punto por corte
fig.add_trace(go.Scatter(x=s['corte'].astype(str), y=s['ganancia'], mode='markers+lines',
                         name='semillerío', line=dict(color='black', dash='dash')))

fig.update_layout(
    title=f"Ganancia por corte - {resultados_cortes['experimento'].iloc[0]}",
    xaxis_title='Cantidad de clientes estimulados',
    yaxis_title='Ganancia en Octubre',
    template='plotly_white'
)
fig.show()

##### Mejor Corte por Modelo

In [27]:
fig = go.Figure(go.Bar(
    x=resultados_mejorN['modelo'], y=resultados_mejorN['mejor_corte'],
    text=[f'{p:.2f}%<br>{g/1e6:.1f}M' for p, g in zip(resultados_mejorN['pct_clientes'], resultados_mejorN['ganancia_maxima'])],   # CAMBIO: % de clientes y ganancia máxima sobre cada barra
    marker_color=['black' if m == 'semillerio' else 'steelblue' for m in resultados_mejorN['modelo']],
))
fig.update_layout(
    title=f"Mejor corte por modelo - {resultados_mejorN['experimento'].iloc[0]}",
    xaxis_title='Modelo', yaxis_title='Mejor corte (clientes)',
    template='plotly_white'
)
fig.show()

#### Resultados de la Hipótesis 1 -- train solo con el mes más cercano, validación y test con BAJA+2 --

**Diseño:** optimización con train 201906 (target BAJA+1 y BAJA+2 contra CONTINUA) y validación 201908 con AUC contra BAJA+2 solamente (543 positivos), 50 trials. Reentrenamiento con 201908 (target fusionado) y las iteraciones del mejor trial, 5 semillas. Test en 201910 con la ganancia contra BAJA+2 solamente. Todo lo demás, igual que el baseline Val-B2 (z803).

**Optimización**
- Mejor trial: 41, con AUC de validación 0.9063 y 367 iteraciones (baseline: 0.9151, misma validación).
- Hiperparámetros: `num_leaves` 35, `learning_rate` 0.015, `min_data_in_leaf` 1151, `feature_fraction` 0.39, `bagging_fraction` 0.90. Árboles mucho más chicos que en el baseline (212 hojas).
- Importancias más repartidas que en el baseline: `bagging_fraction` (0.26), `num_leaves` (0.25), `feature_fraction` (0.21), `min_data_in_leaf` (0.18), `learning_rate` (0.09).
- El `learning_rate` volvió a quedar cerca del borde inferior del rango (0.01).

**Test en 201910 (mediana de 5 semillas; semillerío)**
- AUC contra BAJA+2: 0.9187 (0.9175 a 0.9195); semillerío 0.9198. Baseline: 0.9239 (0.9229 a 0.9241). Los rangos no se superponen.
- Ganancia con umbral 0.025: 191,98 M (media 191,5 M ± 7,0 M); semillerío 193,66 M. Baseline: 257,26 M.
- Ganancia en 4.000 clientes: media 278,3 M (baseline 291,9 M).
- Mejor corte: mediana 4.282 (3,16%), con ganancia máxima de 279,40 M (baseline: 4.170 y 293,15 M).

**Conclusión:** entrenar solo con el mes más cercano empeora el AUC y la ganancia respecto del baseline; la hipótesis no se sostiene con este diseño. La pérdida con el umbral 0.025 (≈ 65 M) es mucho mayor que en los cortes fijos (≈ 14 M): posiblemente las probabilidades quedan en otra escala y el umbral manda otra cantidad de clientes (hipótesis, no comprobada).

**Limitaciones**
- Una sola optimización con una sola semilla de TPE y 50 trials.
- El train tiene un solo mes (128.370 filas en optimización, 131.775 en test), un cuarto del baseline; los rangos de hiperparámetros quedaron iguales a los del baseline.
- Con 5 semillas, el corte sugerido es poco preciso (desvío de 3 a 8 M por corte).

